<a href="https://colab.research.google.com/github/gauravd12345/language_models/blob/main/transformer/transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

dataset = "hf://datasets/PaulineSanchez/Translation_words_and_sentences_english_french/data/train-00000-of-00001-3d14582ea46e1b17.parquet"
df = pd.read_parquet(dataset)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

!pip install sentencepiece
import sentencepiece as spm

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


device: cuda


In [2]:
c1, c2 = df.columns
enc = np.array(df[c1])
dec = np.array(df[c2])

for i in range(len(enc)):
  enc[i] = enc[i].lower()
  dec[i] = dec[i].lower()

for i in np.random.choice(np.arange(len(df)), 10):
  print(f"{enc[i]:<50} | {dec[i]}")

he glanced at her.                                 | il jeta un coup d'œil sur elle.
i can't believe your mom let you go.               | je n'arrive pas à croire que ta mère t'aie laissé t'en aller.
i'm sorry i shot you.                              | je suis désolé de t'avoir tiré dessus.
we were outnumbered.                               | nous avons été surpassées en nombre.
they said yes.                                     | ils ont dit oui.
i had a strange experience last night.             | j'ai fait une étrange expérience, hier au soir.
he asked her to marry him and she accepted his proposal. | il lui a demandé de l'épouser et elle a accepté sa demande.
how many brothers do you have?                     | combien de frères avez-vous ?
it seemed that he was short of money.              | il semblait qu'il manquait d'argent.
staten island is one of the five boroughs of new york. | staten island est l'un des cinq arrondissements de new york.


In [3]:
with open("bpe_train.txt", "w", encoding="utf-8") as f:
    for s in enc:
        f.write(str(s).lower() + "\n")
    for s in dec:
        f.write(str(s).lower() + "\n")

spm.SentencePieceTrainer.train(
    input="bpe_train.txt",
    model_prefix="bpe",
    vocab_size=8000,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

sp = spm.SentencePieceProcessor()
sp.load("bpe.model")

True

In [4]:
""" Hyperparameters """

embedding_dim = 300
hidden_size = 512 # context vector length
batch_size = 128
lr = 0.001
epochs = 10

d_k = torch.tensor(embedding_dim)

pad_tok = sp.pad_id()
unk_tok = sp.unk_id()
sos_tok = sp.bos_id()
eos_tok = sp.eos_id()

vocab_len = sp.get_piece_size()
enc_vocab_len = vocab_len
dec_vocab_len = vocab_len

In [5]:
E = []
for sentence in enc:
    ids = sp.encode(str(sentence).lower(), out_type=int)
    ids = [sos_tok] + ids + [eos_tok]
    E.append(torch.tensor(ids, dtype=torch.long))

D = []
for sentence in dec:
    ids = sp.encode(str(sentence).lower(), out_type=int)
    ids = [sos_tok] + ids + [eos_tok]
    D.append(torch.tensor(ids, dtype=torch.long))

enc_pad_tok = pad_tok
dec_pad_tok = pad_tok

max_encoder_seq = max(len(seq) for seq in E)
max_decoder_seq = max(len(seq) for seq in D)

In [6]:
class TransformerDataset(Dataset):
  def __init__(self, e, d):
    self.e = e
    self.d = d

  def __getitem__(self, index):
     return self.e[index], self.d[index]

  def __len__(self):
    return len(self.e)

""" collate function for batching """
def collate_fn(batch):
  e, d = zip(*batch)

  max_e = max(len(seq) for seq in e)
  max_d = max(len(seq) for seq in d)

  pad_e = torch.zeros(len(e), max_e, dtype=torch.long)
  pad_d = torch.zeros(len(d), max_d, dtype=torch.long)

  for i in range(len(e)):
    pad_e[i] = torch.cat([e[i], torch.tensor([enc_pad_tok] * (max_e - len(e[i])))])

  for i in range(len(d)):
    pad_d[i] = torch.cat([d[i], torch.tensor([dec_pad_tok] * (max_d - len(d[i])))])

  return pad_e, pad_d

In [7]:
class Transformer(nn.Module):
  def __init__(self):
    super().__init__()

    self.enc_embed = nn.Embedding(enc_vocab_len, embedding_dim)
    self.enc_pos_embed = nn.Embedding(max_encoder_seq, embedding_dim) # position embedding

    self.dec_embed = nn.Embedding(dec_vocab_len, embedding_dim)
    self.dec_pos_embed = nn.Embedding(max_decoder_seq, embedding_dim) # position embedding

    self.enc_Q = nn.Linear(embedding_dim, embedding_dim)
    self.enc_K = nn.Linear(embedding_dim, embedding_dim)
    self.enc_V = nn.Linear(embedding_dim, embedding_dim)

    self.dec_Q = nn.Linear(embedding_dim, embedding_dim)
    self.dec_K = nn.Linear(embedding_dim, embedding_dim)
    self.dec_V = nn.Linear(embedding_dim, embedding_dim)

    self.cross_Q = nn.Linear(embedding_dim, embedding_dim)
    self.cross_K = nn.Linear(embedding_dim, embedding_dim)
    self.cross_V = nn.Linear(embedding_dim, embedding_dim)

    self.enc_l1 = nn.Linear(embedding_dim, embedding_dim)
    self.enc_l2 = nn.Linear(embedding_dim, embedding_dim)

    self.dec_l1 = nn.Linear(embedding_dim, embedding_dim)
    self.dec_l2 = nn.Linear(embedding_dim, embedding_dim)

    self.fc = nn.Linear(embedding_dim, dec_vocab_len)

    # layernorm
    self.enc_norm1 = nn.LayerNorm(embedding_dim)
    self.enc_norm2 = nn.LayerNorm(embedding_dim)
    self.dec_norm1 = nn.LayerNorm(embedding_dim)
    self.dec_norm2 = nn.LayerNorm(embedding_dim)
    self.dec_norm3 = nn.LayerNorm(embedding_dim)

  def self_attention(self, Q, K, V, x, dec_embed=False, use_mask=False):
    positions = torch.arange(x.size(1)).to(device)
    if dec_embed == True:
      positions = self.dec_pos_embed(positions)
      E_x = self.dec_embed(x) + positions # word embedding
    else:
      positions = self.enc_pos_embed(positions)
      E_x = self.enc_embed(x) + positions # word embedding

    n = E_x.size(1)

    query = Q(E_x)
    key = K(E_x)
    value = V(E_x)

    dot = torch.matmul(query, key.mT) / (d_k ** 0.5) # scaled dot product
    if use_mask == True:
      mask = torch.tril(torch.ones(n, n, dtype=torch.float)).to(device) # masking future inputs
      mask = mask.masked_fill(mask == 0, float('-inf'))
      mask = mask.masked_fill(mask == 1, 0)

      dot = dot + mask

    wei = torch.softmax(dot, dim=-1) # alignment weights between words
    wei_value = torch.matmul(wei, value) # weighted values
    return wei_value, E_x  # also return E_x for residual connection

  def cross_attention(self, Q, K, V, h, z):
    query = Q(z)
    key = K(h)
    value = V(h)

    dot = torch.matmul(query, key.mT) / (d_k ** 0.5) # scaled dot product

    wei = torch.softmax(dot, dim=-1) # alignment weights between words
    wei_value = torch.matmul(wei, value) # weighted values

    return wei_value

  def encoder(self, x):
    wei_value, E_x = self.self_attention(self.enc_Q, self.enc_K, self.enc_V, x)
    wei_value = self.enc_norm1(E_x + wei_value)       # add & norm

    enc_out = self.enc_l1(wei_value) # output FFN
    enc_out = torch.relu(enc_out)
    enc_out = self.enc_l2(enc_out)
    enc_out = self.enc_norm2(wei_value + enc_out)     # add & norm

    return enc_out

  def decoder(self, x, enc_out):
    wei_value, E_x = self.self_attention(self.dec_Q, self.dec_K, self.dec_V, x, dec_embed=True, use_mask=True)
    wei_value = self.dec_norm1(E_x + wei_value)       # add & norm

    cross = self.cross_attention(self.cross_Q, self.cross_K, self.cross_V, enc_out, wei_value)
    cross = self.dec_norm2(wei_value + cross)         # add & norm

    out = self.dec_l1(cross) # output FFN
    out = torch.relu(out)
    out = self.dec_l2(out)
    out = self.dec_norm3(cross + out)                 # add & norm

    return out

  def forward(self, e, d=None):
    enc_out = self.encoder(e)
    dec_out = self.decoder(d, enc_out)

    out = self.fc(dec_out) # output FFN
    return out

In [8]:
transformer = Transformer().to(device)

dataset = TransformerDataset(E, D)
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

criterion = nn.CrossEntropyLoss(ignore_index=dec_pad_tok)

optimizer = optim.Adam(
    transformer.parameters(),
    lr=lr
)

transformer.train()

for epoch in range(epochs):
    total_loss = 0.0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")

    for e, d in progress_bar:
        e = e.to(device)
        d = d.to(device)

        optimizer.zero_grad()

        d_in = d[:, :-1]
        d_out = d[:, 1:]

        out = transformer(e, d_in)

        loss = criterion(
            out.reshape(-1, out.size(-1)), d_out.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch: {epoch+1}/{epochs} | loss: {avg_loss:.4f}")

Epoch 1/10: 100%|██████████| 1371/1371 [00:52<00:00, 26.26it/s, loss=1.69]


Epoch: 1/10 | loss: 2.4965


Epoch 2/10: 100%|██████████| 1371/1371 [00:55<00:00, 24.72it/s, loss=1.48]


Epoch: 2/10 | loss: 1.3628


Epoch 3/10: 100%|██████████| 1371/1371 [00:53<00:00, 25.73it/s, loss=1.11]


...

Epoch: 8/10 | loss: 0.6779


Epoch 9/10: 100%|██████████| 1371/1371 [00:52<00:00, 25.97it/s, loss=0.59]


Epoch: 9/10 | loss: 0.6374


Epoch 10/10: 100%|██████████| 1371/1371 [00:52<00:00, 25.91it/s, loss=0.656]

Epoch: 10/10 | loss: 0.6016


In [10]:
transformer.eval()

text = """
Yesterday I went to the market with my sister. We bought fresh bread, fruit, and some cheese for dinner.
The weather was cold, but the streets were full of people. After lunch, we visited a small bookstore near the river and
spent almost an hour looking at old novels. I was tired when I got home, but I still watched a movie before going to sleep.
"""

text = sent_tokenize(text)

for test in text:
    seq = sp.encode(test.lower(), out_type=int)
    seq = [sos_tok] + seq + [eos_tok]

    with torch.no_grad():
        enc_input = torch.tensor(seq, device=device).unsqueeze(0)  # encoder input

        # start decoder with just <sos>
        dec_input = torch.tensor([[sos_tok]], device=device)

        pred_ids = []

        for _ in range(max_decoder_seq):
            out = transformer(enc_input, dec_input)  # (1, current_len, vocab)

            next_tok = out[0, -1, :].argmax().item()  # take last predicted token

            if next_tok == eos_tok:
                break

            if next_tok not in [sos_tok, dec_pad_tok]:
                pred_ids.append(next_tok)

            # append predicted token to decoder input for next step
            next_tok_tensor = torch.tensor([[next_tok]], device=device)
            dec_input = torch.cat([dec_input, next_tok_tensor], dim=1)

    print(sp.decode(pred_ids))

hier, je suis allé au marché avec ma sœur.
nous avons acheté du pain, de fruits, de le chéton pour le dîner.
le temps était froid, mais les gens étaient noyés de la rue.
après le déjeuner, nous visité une petite librai près de la rivière à la rivière à la maison.
j'étais fatigué quand je suis encore chez moi, mais je vais me mettre un film.
